# Fase 2: Calidad de Datos — Proyecto Pharma Sales Forecast

Este notebook realiza la auditoría y limpieza de calidad de datos sobre el dataset integrado de ventas, previo a las fases de Análisis Exploratorio (EDA) y Preparación.

## Objetivos de Calidad:
1. **Estandarización de nombres de columnas**: Convertir a minúsculas, remover acentos y espacios, y estandarizar a snake_case.
2. **Revisión y corrección de tipos**: Castear campos numéricos y categóricos.
3. **Completar y corregir campos de fecha**: Asegurar consistencia entre `date`, `year`, `month` y `weekday_name`.
4. **Validación de reglas lógicas**: Asegurar que la variable `hour` solo contenga valor en la granularidad horaria, asignando nulo (NA) en el resto.
5. **Auditoría de duplicados y valores ausentes**.
6. **Guardado del dataset limpio** para las fases posteriores.

In [ ]:
from pathlib import Path
import re
import unicodedata
from io import StringIO
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parent
INPUT_PATH = PROJECT_ROOT / "02_datos/03_Entrenamiento/01_train_tablon_integrado.pkl"
OUTPUT_PATH = PROJECT_ROOT / "02_datos/03_Entrenamiento/02_train_tablon_calidad.pkl"
REPORT_PATH = PROJECT_ROOT / "06_resultados/Calidad_Datos/informe_calidad_datos.md"

print(f"Ruta de entrada: {INPUT_PATH}")
print(f"Ruta de salida: {OUTPUT_PATH}")

### 1. Carga del Tablón Integrado

In [ ]:
df = pd.read_pickle(INPUT_PATH)
print(f"Dataset cargado. Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
df.head()

### 2. Normalización de Nombres de Columnas

In [ ]:
def normalize_column_name(col: str) -> str:
    col = str(col).strip().lower()
    col = unicodedata.normalize("NFKD", col)
    col = "".join(ch for ch in col if not unicodedata.combining(ch))
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col

original_columns = list(df.columns)
normalized_columns = [normalize_column_name(c) for c in df.columns]

if len(set(normalized_columns)) != len(normalized_columns):
    raise ValueError(f"Colisiones de nombres tras normalizar: {normalized_columns}")

df_clean = df.copy()
df_clean.columns = normalized_columns
print("Nombres de columnas normalizados:")
for old, new in zip(original_columns, normalized_columns):
    print(f"  {old} -> {new}")

### 3. Corrección de Tipos y Tratamiento Categórico

In [ ]:
# Conversión a numérico para columnas temporales
for col in ["year", "month", "hour"]:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# Estandarización de cadenas de texto
df_clean["granularity"] = df_clean["granularity"].astype(str).str.strip().str.lower()
df_clean["weekday_name"] = df_clean["weekday_name"].astype(str).str.strip().str.title()
df_clean["weekday_name"] = df_clean["weekday_name"].replace({"Nan": pd.NA, "None": pd.NA})

print("Tipos de datos actualizados:")
print(df_clean[["year", "month", "hour", "granularity", "weekday_name"]].dtypes)

### 4. Correcciones Lógicas y Consistencia de Fechas

In [ ]:
df_clean["date"] = pd.to_datetime(df_clean["date"], errors="coerce")

# Imputar year y month desde date si son nulos
df_clean["year"] = df_clean["year"].astype("Int64").fillna(df_clean["date"].dt.year.astype("Int64"))
df_clean["month"] = df_clean["month"].astype("Int64").fillna(df_clean["date"].dt.month.astype("Int64"))

# Imputar weekday_name desde date si es nulo
df_clean["weekday_name"] = df_clean["weekday_name"].replace({"<NA>": pd.NA})
df_clean["weekday_name"] = df_clean["weekday_name"].fillna(df_clean["date"].dt.day_name())

# Validación de la hora según la granularidad
df_clean["hour"] = df_clean["hour"].astype("Int64")
hourly_mask = df_clean["granularity"].eq("hour")
df_clean.loc[hourly_mask, "hour"] = df_clean.loc[hourly_mask, "hour"].fillna(df_clean.loc[hourly_mask, "date"].dt.hour.astype("Int64"))
df_clean.loc[~hourly_mask, "hour"] = pd.NA

print("Corrección lógica de fecha completada.")

### 5. Auditoría de Calidad: Duplicados y Nulos

In [ ]:
duplicate_count = int(df_clean.duplicated().sum())
missing_summary = df_clean.isna().sum()

print(f"Filas completamente duplicadas: {duplicate_count}")
print("\nResumen de valores faltantes finales:")
print(missing_summary)

### 6. Guardado del Dataset y Reporte

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_pickle(OUTPUT_PATH)
print(f"Dataframe de calidad guardado en: {OUTPUT_PATH}")